<a href="https://colab.research.google.com/github/shawdaena/Parallel-Processing-and-Distributed-System-Lab/blob/main/Searching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#all string are match then show the result
%%writefile search_phonebook.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>

#include <cuda_runtime.h>

using namespace std;

#define MAX_STR_LEN 50

// CUDA device function to perform substring check
__device__ bool check(char* str1, char* str2, int len) {
    for (int i = 0; str1[i] != '\0'; i++) {
        int j = 0;
        while (str1[i + j] != '\0' &&
               str2[j] != '\0' &&
               str1[i + j] == str2[j]) {
            j++;
        }
        if (j == len - 1) {
            return true;
        }
    }
    return false;
}

// CUDA kernel
__global__ void searchPhonebook(char* d_names, char* d_numbers,
                                int num_contacts,
                                char* search_name,
                                int name_length) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_contacts) {
        char* current_name   = d_names   + (idx * MAX_STR_LEN);
        char* current_number = d_numbers + (idx * MAX_STR_LEN);

        if (check(current_name, search_name, name_length)) {
            printf("%s %s\n", current_name, current_number);
        }
    }
}

int main(int argc, char* argv[]) {

    if (argc != 3) {
        cerr << "Usage: " << argv[0]
             << " <search_name> <num_threads>" << endl;
        return 1;
    }

    string search_string = argv[1];
    int num_threads = atoi(argv[2]);
    string file_name = "/content/phonebook1.txt";

    // Read file
    vector<string> raw_lines;
    ifstream file(file_name);

    if (!file.is_open()) {
        cerr << "Error opening file: " << file_name << endl;
        return 1;
    }

    string line;
    while (getline(file, line)) {
        if (!line.empty())
            raw_lines.push_back(line);
    }
    file.close();

    int num_contacts = raw_lines.size();

    // Host memory
    char* h_names   = (char*)malloc(num_contacts * MAX_STR_LEN);
    char* h_numbers = (char*)malloc(num_contacts * MAX_STR_LEN);

    for (int i = 0; i < num_contacts; i++) {
        string current_line = raw_lines[i];
        int pos = current_line.find(",");

        string name   = current_line.substr(1, pos - 2);
        string number = current_line.substr(pos + 2,
                                            current_line.size() - pos - 3);

        strncpy(h_names   + (i * MAX_STR_LEN),
                name.c_str(), MAX_STR_LEN - 1);
        strncpy(h_numbers + (i * MAX_STR_LEN),
                number.c_str(), MAX_STR_LEN - 1);

        h_names[(i * MAX_STR_LEN) + MAX_STR_LEN - 1] = '\0';
        h_numbers[(i * MAX_STR_LEN) + MAX_STR_LEN - 1] = '\0';
    }

    // Device memory
    char *d_names, *d_numbers, *d_search_name;
    int name_len = search_string.length() + 1;

    cudaMalloc((void**)&d_names,   num_contacts * MAX_STR_LEN);
    cudaMalloc((void**)&d_numbers, num_contacts * MAX_STR_LEN);
    cudaMalloc((void**)&d_search_name, name_len);

    cudaMemcpy(d_names, h_names,
               num_contacts * MAX_STR_LEN,
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_numbers, h_numbers,
               num_contacts * MAX_STR_LEN,
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_search_name, search_string.c_str(),
               name_len, cudaMemcpyHostToDevice);

    // Kernel launch (same as your logic)
    for (int i = 0; i < num_contacts; i += num_threads) {

        int thread_count = min(num_contacts - i, num_threads);

        searchPhonebook<<<1, thread_count>>>(
            d_names   + (i * MAX_STR_LEN),
            d_numbers + (i * MAX_STR_LEN),
            thread_count,
            d_search_name,
            name_len
        );

        cudaDeviceSynchronize();
    }

    // Cleanup
    free(h_names);
    free(h_numbers);
    cudaFree(d_names);
    cudaFree(d_numbers);
    cudaFree(d_search_name);

    return 0;
}


Writing search_phonebook.cu


In [ ]:
!nvcc -arch=sm_75 search_phonebook.cu -o search_phonebook

In [ ]:
!time ./search_phonebook TAMANNA 100 > output1.txt


real	0m0.163s
user	0m0.015s
sys	0m0.110s


In [ ]:
#Accendaing order
%%writefile search_phonebook.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>

#include <cuda_runtime.h>

using namespace std;

#define MAX_STR_LEN 50

// Struct for sorted result
struct ResultContact {
    string name;
    string number;

    // Ascending sort by name
    bool operator<(const ResultContact& other) const {
        return name < other.name;
    }
};

// Device substring check
__device__ bool check(char* str1, char* str2, int len) {
    for (int i = 0; str1[i] != '\0'; i++) {
        int j = 0;
        while (str1[i + j] != '\0' &&
               str2[j] != '\0' &&
               str1[i + j] == str2[j]) {
            j++;
        }
        if (j == len - 1) {
            return true;
        }
    }
    return false;
}

// Kernel: mark matches
__global__ void searchPhonebook(char* d_names,
                                int num_contacts,
                                char* search_name,
                                int name_length,
                                int* d_results) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_contacts) {
        char* current_name = d_names + (idx * MAX_STR_LEN);
        d_results[idx] = check(current_name, search_name, name_length) ? 1 : 0;
    }
}

int main(int argc, char* argv[]) {

    if (argc != 3) {
        cerr << "Usage: " << argv[0]
             << " <search_name> <num_threads>" << endl;
        return 1;
    }

    string search_string = argv[1];
    int num_threads = atoi(argv[2]);
    string file_name = "/content/phonebook1.txt";

    // Read phonebook
    vector<string> host_names;
    vector<string> host_numbers;

    ifstream file(file_name);
    if (!file.is_open()) {
        cerr << "Error opening file: " << file_name << endl;
        return 1;
    }

    string line;
    while (getline(file, line)) {
        if (line.empty()) continue;
        int pos = line.find(",");
        string name = line.substr(1, pos - 2);
        string number = line.substr(pos + 2, line.size() - pos - 3);
        host_names.push_back(name);
        host_numbers.push_back(number);
    }
    file.close();

    int num_contacts = host_names.size();

    // Flatten names for CUDA
    char* h_names = (char*)malloc(num_contacts * MAX_STR_LEN);
    int* h_results = (int*)malloc(num_contacts * sizeof(int));

    for (int i = 0; i < num_contacts; i++) {
        strncpy(h_names + (i * MAX_STR_LEN),
                host_names[i].c_str(),
                MAX_STR_LEN - 1);
        h_names[i * MAX_STR_LEN + MAX_STR_LEN - 1] = '\0';
    }

    // Device memory
    char *d_names, *d_search_name;
    int *d_results;
    int search_len = search_string.length() + 1;

    cudaMalloc((void**)&d_names, num_contacts * MAX_STR_LEN);
    cudaMalloc((void**)&d_results, num_contacts * sizeof(int));
    cudaMalloc((void**)&d_search_name, search_len);

    cudaMemcpy(d_names, h_names,
               num_contacts * MAX_STR_LEN,
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_search_name, search_string.c_str(),
               search_len, cudaMemcpyHostToDevice);

    // Launch kernel
    int blocks = (num_contacts + num_threads - 1) / num_threads;

    searchPhonebook<<<blocks, num_threads>>>(
        d_names,
        num_contacts,
        d_search_name,
        search_len,
        d_results
    );
    cudaDeviceSynchronize();

    // Copy results back
    cudaMemcpy(h_results, d_results,
               num_contacts * sizeof(int),
               cudaMemcpyDeviceToHost);

    // Collect matched contacts
    vector<ResultContact> matched_contacts;
    for (int i = 0; i < num_contacts; i++) {
        if (h_results[i] == 1) {
            matched_contacts.push_back(
                {host_names[i], host_numbers[i]}
            );
        }
    }

    // Sort ascending by name
    sort(matched_contacts.begin(), matched_contacts.end());

    // Print output
    cout << "Search Results (Ascending Order):" << endl;
    for (const auto& c : matched_contacts) {
        cout << c.name << " " << c.number << endl;
    }

    // Cleanup
    free(h_names);
    free(h_results);
    cudaFree(d_names);
    cudaFree(d_results);
    cudaFree(d_search_name);

    return 0;
}


Overwriting search_phonebook.cu


In [ ]:
!nvcc -arch=sm_75 search_phonebook.cu -o search_phonebook

In [ ]:
!time ./search_phonebook TAM 100 > output1.txt


real	0m0.130s
user	0m0.014s
sys	0m0.112s


In [ ]:
#Decendaing order
%%writefile search_phonebook.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>

#include <cuda_runtime.h>

using namespace std;

#define MAX_STR_LEN 50

// Struct for sorted result
struct ResultContact {
    string name;
    string number;

    // Ascending sort by name
    bool operator<(const ResultContact& other) const {
        return name > other.name;
    }
};

// Device substring check
__device__ bool check(char* str1, char* str2, int len) {
    for (int i = 0; str1[i] != '\0'; i++) {
        int j = 0;
        while (str1[i + j] != '\0' &&
               str2[j] != '\0' &&
               str1[i + j] == str2[j]) {
            j++;
        }
        if (j == len - 1) {
            return true;
        }
    }
    return false;
}

// Kernel: mark matches
__global__ void searchPhonebook(char* d_names,
                                int num_contacts,
                                char* search_name,
                                int name_length,
                                int* d_results) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_contacts) {
        char* current_name = d_names + (idx * MAX_STR_LEN);
        d_results[idx] = check(current_name, search_name, name_length) ? 1 : 0;
    }
}

int main(int argc, char* argv[]) {

    if (argc != 3) {
        cerr << "Usage: " << argv[0]
             << " <search_name> <num_threads>" << endl;
        return 1;
    }

    string search_string = argv[1];
    int num_threads = atoi(argv[2]);
    string file_name = "/content/phonebook1.txt";

    // Read phonebook
    vector<string> host_names;
    vector<string> host_numbers;

    ifstream file(file_name);
    if (!file.is_open()) {
        cerr << "Error opening file: " << file_name << endl;
        return 1;
    }

    string line;
    while (getline(file, line)) {
        if (line.empty()) continue;
        int pos = line.find(",");
        string name = line.substr(1, pos - 2);
        string number = line.substr(pos + 2, line.size() - pos - 3);
        host_names.push_back(name);
        host_numbers.push_back(number);
    }
    file.close();

    int num_contacts = host_names.size();

    // Flatten names for CUDA
    char* h_names = (char*)malloc(num_contacts * MAX_STR_LEN);
    int* h_results = (int*)malloc(num_contacts * sizeof(int));

    for (int i = 0; i < num_contacts; i++) {
        strncpy(h_names + (i * MAX_STR_LEN),
                host_names[i].c_str(),
                MAX_STR_LEN - 1);
        h_names[i * MAX_STR_LEN + MAX_STR_LEN - 1] = '\0';
    }

    // Device memory
    char *d_names, *d_search_name;
    int *d_results;
    int search_len = search_string.length() + 1;

    cudaMalloc((void**)&d_names, num_contacts * MAX_STR_LEN);
    cudaMalloc((void**)&d_results, num_contacts * sizeof(int));
    cudaMalloc((void**)&d_search_name, search_len);

    cudaMemcpy(d_names, h_names,
               num_contacts * MAX_STR_LEN,
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_search_name, search_string.c_str(),
               search_len, cudaMemcpyHostToDevice);

    // Launch kernel
    int blocks = (num_contacts + num_threads - 1) / num_threads;

    searchPhonebook<<<blocks, num_threads>>>(
        d_names,
        num_contacts,
        d_search_name,
        search_len,
        d_results
    );
    cudaDeviceSynchronize();

    // Copy results back
    cudaMemcpy(h_results, d_results,
               num_contacts * sizeof(int),
               cudaMemcpyDeviceToHost);

    // Collect matched contacts
    vector<ResultContact> matched_contacts;
    for (int i = 0; i < num_contacts; i++) {
        if (h_results[i] == 1) {
            matched_contacts.push_back(
                {host_names[i], host_numbers[i]}
            );
        }
    }

    // Sort ascending by name
    sort(matched_contacts.begin(), matched_contacts.end());

    // Print output
    cout << "Search Results (Ascending Order):" << endl;
    for (const auto& c : matched_contacts) {
        cout << c.name << " " << c.number << endl;
    }

    // Cleanup
    free(h_names);
    free(h_results);
    cudaFree(d_names);
    cudaFree(d_results);
    cudaFree(d_search_name);

    return 0;
}


Overwriting search_phonebook.cu


In [ ]:
!nvcc -arch=sm_75 search_phonebook.cu -o search_phonebook

In [ ]:
!time ./search_phonebook TAM 100 > output1.txt


real	0m0.131s
user	0m0.012s
sys	0m0.115s


In [ ]:
%%writefile search_phonebook_cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#include <iostream>
#include <fstream>
#include <vector>
#include <string>

#include <cuda_runtime.h>

using namespace std;

#define MAX_STR_LEN 256
#define MIN_MATCH 4

/* -------- DEVICE HELPERS -------- */

// Device version of tolower (ASCII only)
__device__ char d_tolower(char c) {
    if (c >= 'A' && c <= 'Z') return c + ('a' - 'A');
    return c;
}

// Device version of strlen
__device__ int d_strlen(const char* s) {
    int len = 0;
    while (s[len] != '\0') len++;
    return len;
}

/* -------- DEVICE STRING MATCH -------- */
__device__ bool check(const char* line, const char* search) {

    int n = d_strlen(line);
    int m = d_strlen(search);

    for (int start = 0; start <= n - MIN_MATCH; start++) {

        // first letter anchor
        if (d_tolower(line[start]) != d_tolower(search[0]))
            continue;

        int k = 0;
        while (start + k < n &&
               k < m &&
               d_tolower(line[start + k]) == d_tolower(search[k])) {
            k++;
        }

        if (k >= MIN_MATCH)
            return true;
    }
    return false;
}

/* -------- CUDA KERNEL -------- */
__global__ void phonebookSearch(char* d_lines,
                                int num_lines,
                                char* d_search,
                                int* d_match_flags) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_lines) {
        char* line = d_lines + idx * MAX_STR_LEN;
        d_match_flags[idx] = check(line, d_search) ? 1 : 0;
    }
}

/* -------- MAIN -------- */
int main(int argc, char* argv[]) {

    if (argc < 3) {
        cerr << "Usage: " << argv[0]
             << " <phonebook_file> <search_term>\n";
        return 1;
    }

    string file_name = argv[1];
    string search_term = argv[2];

    /* -------- READ FILE -------- */
    vector<string> lines;
    ifstream file(file_name);

    if (!file.is_open()) {
        cerr << "Could not open file\n";
        return 1;
    }

    string line;
    while (getline(file, line)) {
        if (!line.empty())
            lines.push_back(line);
    }
    file.close();

    int num_lines = lines.size();

    /* -------- FLATTEN DATA -------- */
    char* h_lines = (char*)malloc(num_lines * MAX_STR_LEN);
    int* h_match_flags = (int*)malloc(num_lines * sizeof(int));

    for (int i = 0; i < num_lines; i++) {
        strncpy(h_lines + i * MAX_STR_LEN,
                lines[i].c_str(),
                MAX_STR_LEN - 1);
        h_lines[i * MAX_STR_LEN + MAX_STR_LEN - 1] = '\0';
    }

    /* -------- DEVICE MEMORY -------- */
    char *d_lines, *d_search;
    int *d_match_flags;

    cudaMalloc(&d_lines, num_lines * MAX_STR_LEN);
    cudaMalloc(&d_match_flags, num_lines * sizeof(int));
    cudaMalloc(&d_search, search_term.size() + 1);

    cudaMemcpy(d_lines, h_lines,
               num_lines * MAX_STR_LEN,
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_search, search_term.c_str(),
               search_term.size() + 1,
               cudaMemcpyHostToDevice);

    /* -------- KERNEL LAUNCH -------- */
    int threads = 256;
    int blocks = (num_lines + threads - 1) / threads;

    phonebookSearch<<<blocks, threads>>>(
        d_lines,
        num_lines,
        d_search,
        d_match_flags
    );
    cudaDeviceSynchronize();

    /* -------- COPY BACK -------- */
    cudaMemcpy(h_match_flags, d_match_flags,
               num_lines * sizeof(int),
               cudaMemcpyDeviceToHost);

    /* -------- WRITE OUTPUT -------- */
    ofstream out("output1.txt");
    for (int i = 0; i < num_lines; i++) {
        if (h_match_flags[i])
            out << lines[i] << "\n";
    }
    out.close();

    printf("Search complete. Results saved to output1.txt\n");

    /* -------- CLEANUP -------- */
    free(h_lines);
    free(h_match_flags);
    cudaFree(d_lines);
    cudaFree(d_match_flags);
    cudaFree(d_search);

    return 0;
}

Overwriting search_phonebook_cuda.cu


In [ ]:
!nvcc -arch=sm_75 search_phonebook_cuda.cu -o search_phonebook

In [ ]:
!time ./search_phonebook phonebook1.txt TAMANNA > output1.txt


real	0m0.129s
user	0m0.013s
sys	0m0.111s


In [ ]:
#phone_Book Searching at any position
%%writefile search_phonebook.cu
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>
#include <cuda_runtime.h>

using namespace std;

// ১. Preprocessing Function
string preprocess(string s) {
    if (s.empty()) return "";
    s.erase(remove(s.begin(), s.end(), '\"'), s.end());
    transform(s.begin(), s.end(), s.begin(), ::tolower);
    size_t first = s.find_first_not_of(" \t\r\n");
    if (string::npos == first) return "";
    size_t last = s.find_last_not_of(" \t\r\n");
    return s.substr(first, (last - first + 1));
}

// ২. CUDA Kernel (LCS Logic)
__global__ void lcs_kernel(char* d_data, int* d_offsets, int* d_lengths, int num_lines,
                           char* d_search_term, int search_len, int* d_scores, int* d_match_pos) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= num_lines) return;

    char* line = d_data + d_offsets[idx];
    int line_len = d_lengths[idx];

    int max_len = 0;
    int end_idx = 0;

    // GPUs local memory DP Calculation (Max search term length: 512)
    int curr[513] = {0};
    int prev[513] = {0};

    for (int i = 0; i < line_len; i++) {
        for (int j = 0; j < search_len; j++) {
            if (line[i] == d_search_term[j]) {
                curr[j + 1] = prev[j] + 1;
                if (curr[j + 1] > max_len) {
                    max_len = curr[j + 1];
                    end_idx = i;
                }
            } else {
                curr[j + 1] = 0;
            }
        }
        for (int j = 0; j <= search_len; j++) {
            prev[j] = curr[j];
        }
    }
    d_scores[idx] = max_len;
    d_match_pos[idx] = (max_len > 0) ? (end_idx - max_len + 1) : 0;
}

struct FinalRes { int score; string line, part; };

int main(int argc, char** argv) {
    // ==========================================
    // Fixed Configuration
    // ==========================================
    string file_path = "/content/phonebook1.txt";
    int threshold = 3;
    // ==========================================

    if (argc < 3) {
        cerr << "Usage: ./search_phonebook <search_term> <threads_per_block>" << endl;
        return 1;
    }

    string search_word = argv[1];
    int threadsPerBlock = stoi(argv[2]); // terminals ১০০ or others

    string search_term = preprocess(search_word);

    ifstream f(file_path);
    if (!f.is_open()) {
        cerr << "Error: File not found at " << file_path << endl;
        return 1;
    }

    vector<string> original_lines, clean_lines;
    string raw_line, all_data_flat = "";
    vector<int> offsets, lengths;

    while (getline(f, raw_line)) {
        if (raw_line.empty()) continue;
        original_lines.push_back(raw_line);
        string cleaned = preprocess(raw_line);
        clean_lines.push_back(cleaned);

        offsets.push_back(all_data_flat.size());
        lengths.push_back(cleaned.size());
        all_data_flat += cleaned;
    }
    f.close();

    int num_lines = clean_lines.size();
    int search_len = search_term.size();

    char *d_data, *d_search_term;
    int *d_offsets, *d_lengths, *d_scores, *d_match_pos;

    cudaMalloc(&d_data, all_data_flat.size());
    cudaMalloc(&d_offsets, num_lines * sizeof(int));
    cudaMalloc(&d_lengths, num_lines * sizeof(int));
    cudaMalloc(&d_search_term, search_len);
    cudaMalloc(&d_scores, num_lines * sizeof(int));
    cudaMalloc(&d_match_pos, num_lines * sizeof(int));

    cudaMemcpy(d_data, all_data_flat.c_str(), all_data_flat.size(), cudaMemcpyHostToDevice);
    cudaMemcpy(d_offsets, offsets.data(), num_lines * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_lengths, lengths.data(), num_lines * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_search_term, search_term.c_str(), search_len, cudaMemcpyHostToDevice);

    // Determine the number of grides needed
    int blocksPerGrid = (num_lines + threadsPerBlock - 1) / threadsPerBlock;

    lcs_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_data, d_offsets, d_lengths, num_lines, d_search_term, search_len, d_scores, d_match_pos);
    cudaDeviceSynchronize();

    vector<int> h_scores(num_lines), h_pos(num_lines);
    cudaMemcpy(h_scores.data(), d_scores, num_lines * sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(h_pos.data(), d_match_pos, num_lines * sizeof(int), cudaMemcpyDeviceToHost);

    vector<FinalRes> results;
    for (int i = 0; i < num_lines; i++) {
        if (h_scores[i] >= threshold) {
            results.push_back({h_scores[i], original_lines[i], clean_lines[i].substr(h_pos[i], h_scores[i])});
        }
    }

    sort(results.begin(), results.end(), [](FinalRes a, FinalRes b) { return a.score > b.score; });

    ofstream fout("output.txt");
    for (auto& r : results) {
        fout << "[Score: " << r.score << "] " << r.line << " (Match: " << r.part << ")" << endl;
    }
    fout.close();

    cout << "GPU Search Complete. Threads used per block: " << threadsPerBlock << endl;
    cout << "Total Matches Found: " << results.size() << " (Threshold: " << threshold << ")" << endl;

    cudaFree(d_data); cudaFree(d_offsets); cudaFree(d_lengths);
    cudaFree(d_search_term); cudaFree(d_scores); cudaFree(d_match_pos);

    return 0;
}

Overwriting search_phonebook.cu


In [ ]:
!nvcc -arch=sm_75 search_phonebook.cu -o search_phonebook

In [ ]:
!time ./search_phonebook TAMANNA 50

GPU Search Complete. Threads used per block: 50
Total Matches Found: 84 (Threshold: 3)

real	0m0.178s
user	0m0.012s
sys	0m0.116s
